# Lab type: review
# Course: ML401 — MLOps & Model Deployment
# Lesson: Kubernetes for ML
# Task: Review the Kubernetes manifests below and answer the open-ended questions about deployment strategy, resource configuration, and probe design for a production ML serving workload.

## The workload

You are deploying a churn prediction model with the following characteristics:

- Model artefact size: **800MB** (loads into ~1.2GB RAM when deserialised)
- Startup time: **~50 seconds** (model deserialisation)
- Prediction latency: **~30ms** per request under normal load
- Expected traffic: **200 req/s** peak, **40 req/s** off-peak
- Each prediction request uses ~100ms of CPU
- SLA: **P99 latency < 200ms**, **availability > 99.9%**

## The manifest to review

```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: churn-model
  namespace: ml-serving
spec:
  replicas: 3
  selector:
    matchLabels:
      app: churn-model
  template:
    metadata:
      labels:
        app: churn-model
    spec:
      containers:
      - name: churn-model
        image: registry.example.com/churn-model:v2.1.0
        ports:
        - containerPort: 8080
        resources:
          requests:
            memory: "1Gi"
            cpu: "500m"
          limits:
            memory: "1500Mi"
            cpu: "2000m"
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 10
          periodSeconds: 10
          failureThreshold: 3
        livenessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 15
          periodSeconds: 30
          failureThreshold: 3
---
apiVersion: autoscaling/v2
kind: HorizontalPodAutoscaler
metadata:
  name: churn-model-hpa
  namespace: ml-serving
spec:
  scaleTargetRef:
    apiVersion: apps/v1
    kind: Deployment
    name: churn-model
  minReplicas: 3
  maxReplicas: 10
  metrics:
  - type: Resource
    resource:
      name: cpu
      target:
        type: Utilization
        averageUtilization: 70
```

## Review question 1: Probe configuration

The model takes ~50 seconds to load. Review the readiness and liveness probe configurations:

```yaml
readinessProbe:
  initialDelaySeconds: 10   # <-- is this correct?
  periodSeconds: 10
  failureThreshold: 3

livenessProbe:
  initialDelaySeconds: 15   # <-- is this correct?
  periodSeconds: 30
  failureThreshold: 3
```

**Answer the following:**

a) What happens to the pod between second 10 and second 50, given the readiness probe configuration above?

b) Is the liveness probe `initialDelaySeconds: 15` safe for a pod that takes 50 seconds to start? What risk does this create?

c) Write corrected probe configurations that account for the 50-second startup time. Justify your chosen values.

**Your answer for Question 1:**

a)

b)

c)
```yaml
# Your corrected probes here:
readinessProbe:

livenessProbe:
```

## Review question 2: Resource limits

The manifest sets:
```yaml
resources:
  requests:
    memory: "1Gi"
    cpu: "500m"
  limits:
    memory: "1500Mi"
    cpu: "2000m"
```

The model loads into ~1.2GB RAM. Peak traffic is 200 req/s; each request uses ~100ms CPU.

**Answer the following:**

a) At peak load, a single pod receives ~67 req/s (200 req/s ÷ 3 pods). Each request uses 100ms CPU. What is the approximate CPU utilisation in millicores per pod at peak load? Is the CPU request (500m) sufficient?

b) The memory request is 1Gi but the model uses 1.2GB. What happens if Kubernetes tries to schedule this pod on a node with only 1Gi of allocatable memory?

c) The memory limit is 1500Mi. If a memory leak causes the pod to use 1600Mi, what happens?

d) Propose revised resource requests and limits based on the workload characteristics. Show your reasoning.

**Your answer for Question 2:**

a)

b)

c)

d)
```yaml
resources:
  requests:
    memory:
    cpu:
  limits:
    memory:
    cpu:
```

## Review question 3: Update strategy

The Deployment above has no `strategy` field, so it uses the Kubernetes default (`RollingUpdate` with `maxSurge: 25%`, `maxUnavailable: 25%`).

With 3 replicas, `maxUnavailable: 25%` rounds down to 0 (since 0.75 < 1). So effectively `maxUnavailable: 0` and `maxSurge: 1` (25% of 3, rounded up).

**Answer the following:**

a) A new model version is being deployed. With the default strategy, how many pods will be running simultaneously at the peak of the rolling update? What is the memory footprint on the node at that point?

b) The new model version (v2.2.0) changes the prediction API: it now returns a confidence interval alongside the prediction score. Existing clients do not send the required new request field. Would you use `RollingUpdate` or `Recreate` for this deployment? Justify your choice.

c) After deploying v2.2.0, prediction latency increases from 30ms to 180ms. You want to roll back immediately. Write the `kubectl` command that would roll back this deployment.

**Your answer for Question 3:**

a)

b)

c) `kubectl command:`

## Question 3c answer

```bash
kubectl rollout undo deployment/churn-model -n ml-serving

# Verify rollback status
kubectl rollout status deployment/churn-model -n ml-serving

# Check which version is now running
kubectl get pods -n ml-serving -o jsonpath='{.items[*].spec.containers[0].image}'
```

Kubernetes stores the previous ReplicaSet. `kubectl rollout undo` switches traffic back to it. The rollback is a rolling operation — it applies the same update strategy in reverse.